# Otimização de Hiperparâmetros com Optuna — Guia Didático Completo

Este notebook é um material de referência para alunos, cobrindo desde a teoria por trás da otimização de hiperparâmetros até a implementação prática com o **Optuna**, do básico ao avançado.

**Ordem do guia:**
1. Introdução teórica
2. A dor da busca manual (motivação)
3. Fundamentos do Optuna: `objective`, `trial`, `study`
4. Trilha prática: **7 exercícios**, do comparativo Grid/Random/Optuna até XGBoost com pruning — escrevendo código de verdade, não só preenchendo números
5. Visualização e diagnóstico
6. Para que servem os hiperparâmetros na prática
7. Resumo final

> Cada exercício tem uma **especificação** (o que a função precisa fazer) em vez de código pronto pra preencher. Tente escrever sozinho antes de abrir o gabarito — é assim que gruda.


## 1. Introdução Teórica: O Problema da Sintonização

A performance de um modelo de Machine Learning depende criticamente dos seus **hiperparâmetros**. Diferente dos parâmetros (pesos), eles não são aprendidos pelo algoritmo — são definidos por você, o cientista de dados, *antes* do treino.

### O que é o Optuna?
É um framework de otimização automática que utiliza a técnica **Tree-structured Parzen Estimator (TPE)**. Diferente do *Grid Search* (que testa tudo e demora muito) ou do *Random Search* (que é puramente sorte), o Optuna "aprende" com as tentativas anteriores para sugerir valores melhores no próximo passo.


In [ ]:
# Instalando as bibliotecas
!pip install optuna scikit-learn matplotlib seaborn xgboost cmaes -q

import optuna
import sklearn

print("Optuna versão:", optuna.__version__)
print("Scikit-learn versão:", sklearn.__version__)


## 2. A "Dor" da Busca Manual

Antes de automatizar, vamos sentir na pele por que isso é necessário. Observe como a tentativa e erro manual é limitada e repetitiva.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score

# Setup dos dados
X, y = load_breast_cancer(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# TESTE MANUAL: Tentando 3 valores diferentes no "chute"
for depth in [2, 10, 50]:
    clf = RandomForestClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_val, clf.predict(X_val))
    print(f"Tentativa com max_depth={depth}: Acurácia = {acc:.4f}")


**Pergunta para reflexão:** e se o melhor `max_depth` fosse 7? Ou 23? Você testaria manualmente todos os valores possíveis? É exatamente esse problema que o Optuna resolve.


## 3. Fundamentos: Anatomia de uma Otimização

Para usar o Optuna, precisamos entender três conceitos-chave:

1. **Objective Function**: é a regra do jogo. Recebe um objeto `trial` (uma tentativa), sugere hiperparâmetros e retorna um valor numérico que queremos maximizar (ex: acurácia) ou minimizar (ex: erro).
2. **Trial**: é uma única execução da função objetivo. Nele usamos os métodos `suggest_int`, `suggest_float` e `suggest_categorical` para amostrar valores dentro de um espaço de busca.
3. **Study**: é a sessão de otimização que gerencia todos os trials, guarda o histórico e sabe qual foi o melhor resultado até agora.

> ⚠️ **Atenção:** a célula abaixo só *define* a função `objective` — rodá-la sozinha não imprime nada na tela, e isso é normal! Ela só é executada de verdade quando chamamos `study.optimize(objective, ...)`, na célula seguinte.


In [ ]:
import optuna
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Carregando dados de exemplo (dataset clássico Iris: 3 espécies de flor, 4 features)
data = load_iris()
X_iris, y_iris = data.data, data.target

# Separamos 80% para treino e 20% para teste/validação
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

def objective(trial):
    # Esta função NÃO roda sozinha! Ela só é chamada quando o Optuna
    # invoca 'study.optimize(objective, ...)' — uma vez por trial.
    # Rodar esta célula apenas DEFINE a função; para ver resultados,
    # é preciso rodar a próxima célula, que cria o study e chama optimize.

    # --- 1. Sugerindo hiperparâmetros para esta tentativa (trial) ---
    # Cada chamada 'suggest_*' pede ao Optuna um valor dentro de um intervalo.
    # A cada trial, esses valores mudam, guiados pelos resultados anteriores.

    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    # ^ nº de árvores da floresta: inteiro entre 10 e 200

    max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
    # ^ profundidade máxima de cada árvore: inteiro entre 2 e 32
    #   log=True porque a diferença entre profundidade 2->4 importa mais
    #   proporcionalmente do que 30->32, então exploramos em escala logarítmica

    min_samples_split = trial.suggest_float('min_samples_split', 0.1, 1.0)
    # ^ fração mínima de amostras exigida para dividir um nó: float entre 0.1 e 1.0

    # --- 2. Criando o modelo com os hiperparâmetros sugeridos ---
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    # --- 3. Treinando o modelo com os dados de treino ---
    clf.fit(X_train_iris, y_train_iris)

    # --- 4. Avaliando no conjunto de teste ---
    preds = clf.predict(X_test_iris)
    accuracy = accuracy_score(y_test_iris, preds)

    # --- 5. Retornando a métrica que o Optuna vai tentar otimizar ---
    return accuracy  # O Optuna tentará MAXIMIZAR este valor (ver direction='maximize' abaixo)


### Executando a Otimização

Criamos um objeto `study` e chamamos o método `optimize`. O parâmetro `direction='maximize'` indica que queremos a maior acurácia possível.


In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("\n--- Melhores Resultados ---")
print(f"Melhor Acurácia: {study.best_value:.4f}")
print(f"Melhores Parâmetros: {study.best_params}")


## 4. Trilha Prática: Do Básico ao Mestre

Agora que você entende a anatomia, é hora de praticar de verdade. São **7 exercícios**. Aviso: a partir do Nível 1, você vai ter que **escrever lógica**, não só preencher números — é assim que se aprende Optuna.

| Nível | Modelo | Foco didático |
|---|---|---|
| 0 | KNN | Grid Search vs Random Search vs Optuna — por que o Optuna vence |
| 1 | ExtraTrees | Escrever a função `objective` inteira a partir de uma especificação |
| 2 | SVM | Decidir sozinho ONDE usar `log=True` e justificar |
| 3 | Random Forest Regressor | Validação cruzada dentro do `objective` + escolher `direction` |
| 4 | ExtraTrees (revisita) | **Sampler Genético (CMA-ES)** vs TPE — outra forma de busca |
| 5 | Gradient Boosting | Espaço de busca **condicional** (define-by-run de verdade) |
| 6 | **XGBoost** | Pruning escrito do zero — o mais desafiador |

### Nível 0: Grid Search vs Random Search vs Optuna (Aquecimento)

Antes de ir direto pro Optuna, vale sentir na prática **por que** ele costuma ser a melhor escolha. Vamos rodar o mesmo problema (KNN) com os três métodos e comparar.

**Grid Search** — força bruta: testa TODAS as combinações da grade.
* Vantagem: simplicidade e bom resultado.
* Desvantagem: inviável em espaços de busca grandes.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.datasets import load_breast_cancer
from scipy.stats import randint
import time

# Dados para o exercício 0 (usados nos 3 métodos, para comparação justa)
X_knn, y_knn = load_breast_cancer(return_X_y=True)
X_train_knn, X_test_knn, y_train_knn, y_test_knn = train_test_split(
    X_knn, y_knn, test_size=0.2, random_state=42
)

# hiperparâmetros do KNN a testar: 4 valores de n_neighbors x 2 tipos de weights = 8 combinações
grid_search = {
    'n_neighbors': [3, 5, 7, 9],   # hiperparâmetro do modelo
    'weights': ['uniform', 'distance']  # hiperparâmetro do modelo
}

# cv=5 -> validação cruzada, não é hiperparâmetro do modelo
# scoring -> métrica de avaliação: 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'r2', 'neg_mean_squared_error'...
grid = GridSearchCV(KNeighborsClassifier(), grid_search, cv=5, scoring='accuracy')

t0 = time.time()
# roda as 8 combinações x 5 folds = 40 treinos
grid.fit(X_train_knn, y_train_knn)
t_grid = time.time() - t0

# melhor combinação encontrada
print(f"[Grid Search] Melhores params: {grid.best_params_}")
print(f"[Grid Search] Melhor score (CV): {grid.best_score_:.4f}")
print(f"[Grid Search] Tempo: {t_grid:.2f}s | Combinações testadas: {len(grid.cv_results_['params'])}")


**Random Search** — semelhante ao Grid Search, mas testa apenas combinações aleatórias, com um número fixo de tentativas (`n_iter`).
* Vantagem: viável para projetos grandes, funciona bem.
* Desvantagem: depende de sorte, tem muita variância.


In [ ]:
# hiperparâmetros do KNN, agora como distribuições (não lista fixa)
param_dist = {
    'n_neighbors': randint(1, 30),              # sorteia inteiros entre 1 e 30
    'weights': ['uniform', 'distance']
}

random_search = RandomizedSearchCV(
    KNeighborsClassifier(), param_dist,
    n_iter=10,              # testa só 10 combinações aleatórias (não todas)
    cv=5, scoring='accuracy', random_state=42
)

t0 = time.time()
random_search.fit(X_train_knn, y_train_knn)
t_random = time.time() - t0

print(f"[Random Search] Melhores params: {random_search.best_params_}")
print(f"[Random Search] Melhor score (CV): {random_search.best_score_:.4f}")
print(f"[Random Search] Tempo: {t_random:.2f}s | Combinações testadas: 10")


**Optuna** — usa os resultados das tentativas anteriores para construir um modelo probabilístico de quais regiões do espaço de hiperparâmetros parecem promissoras (Otimização Bayesiana via TPE por padrão), e escolhe o próximo ponto de forma inteligente.
* Vantagem: eficiente, aprende com as tentativas anteriores.
* Desvantagem: mais complexo, sequencial (pode ser mais lento por tentativa), pode enviesar.

**Sua tarefa:** complete a função objetivo abaixo (mesmo espaço de busca do KNN) e rode o study.


In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective_knn(trial):
    # TODO 1: sugira um INTEIRO para 'n_neighbors' entre 1 e 30
    n_neighbors = trial.suggest_int('n_neighbors', ___, ___)  # <-- preencha os limites

    # TODO 2: sugira uma CATEGORIA para 'weights' entre ['uniform', 'distance']
    weights = trial.suggest_categorical('weights', ___)  # <-- preencha a lista

    # treina o modelo com a combinação sugerida nesta tentativa
    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights)

    # avalia com validação cruzada (5 folds) e tira a média
    score = cross_val_score(model, X_train_knn, y_train_knn, cv=5, scoring='accuracy').mean()
    return score  # métrica que o Optuna tenta maximizar

# cria o estudo: direction='maximize' porque queremos a maior acurácia possível
study_knn = optuna.create_study(direction='maximize')

t0 = time.time()
# roda 10 tentativas, cada uma escolhida com base nos resultados anteriores (mesmo budget do Random Search!)
study_knn.optimize(objective_knn, n_trials=10)
t_optuna = time.time() - t0

print(f"[Optuna] Melhores params: {study_knn.best_params}")
print(f"[Optuna] Melhor score (CV): {study_knn.best_value:.4f}")
print(f"[Optuna] Tempo: {t_optuna:.2f}s | Combinações testadas: 10")


> 💡 **Gabarito** (se travar): `n_neighbors = trial.suggest_int('n_neighbors', 1, 30)` e `weights = trial.suggest_categorical('weights', ['uniform', 'distance'])` — o mesmo espaço de busca usado no Grid e no Random Search acima, pra comparação ficar justa.


### Comparando os três

Com o **mesmo orçamento de tentativas** (10), o Optuna tende a encontrar uma combinação tão boa ou melhor que o Random Search, porque ele **aprende** com cada tentativa em vez de sortear às cegas. E diferente do Grid Search, ele não precisa testar todas as combinações possíveis — o que se torna inviável rapidamente conforme o espaço de busca cresce (imagine 5 hiperparâmetros com 10 valores cada: 100.000 combinações!).


In [ ]:
print("=== Resumo comparativo ===")
print(f"Grid Search   -> score: {grid.best_score_:.4f} | combinações: {len(grid.cv_results_['params'])} | tempo: {t_grid:.2f}s")
print(f"Random Search -> score: {random_search.best_score_:.4f} | combinações: 10 | tempo: {t_random:.2f}s")
print(f"Optuna        -> score: {study_knn.best_value:.4f} | combinações: 10 | tempo: {t_optuna:.2f}s")


> 💡 **Nota didática:** existe também uma quarta família de métodos, os **Algoritmos Genéticos** (ex: CMA-ES), inspirados em seleção natural — eles mantêm uma "população" de combinações que evolui a cada geração. São bons em espaços de busca grandes e complexos, mas são caros computacionalmente porque precisam avaliar uma população inteira a cada geração. O Optuna também suporta CMA-ES como sampler alternativo ao TPE padrão.

---

### Nível 1: Escreva o `objective` do Zero (Fácil)

Chega de preencher números — a partir daqui você escreve a lógica inteira. Abaixo está só a **especificação** do que a função precisa fazer. O import e os dados já estão prontos; o resto é com você.

**Especificação da função `objective_guided(trial)`:**
1. Sugira `n_estimators` (inteiro, entre 50 e 200).
2. Sugira `criterion` (categórico, `'gini'` ou `'entropy'`).
3. Sugira `max_features` (float, entre 0.1 e 1.0).
4. Crie um `ExtraTreesClassifier` com esses três hiperparâmetros e `random_state=42`.
5. Treine com `X_train_et, y_train_et`.
6. Retorne a acurácia calculada em `X_val_et, y_val_et`.

Depois, crie o `study` (pense: acurácia é algo que queremos maximizar ou minimizar?) e rode 15 trials.


In [ ]:
import optuna
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Dados para este exercício (já prontos — não precisa mexer aqui)
X_et, y_et = load_breast_cancer(return_X_y=True)
X_train_et, X_val_et, y_train_et, y_val_et = train_test_split(
    X_et, y_et, test_size=0.2, random_state=42
)

# ESCREVA AQUI a função objective_guided(trial) seguindo a especificação acima.
# Dica: reveja o exemplo do Iris na seção 3 se precisar relembrar a estrutura.



# ESCREVA AQUI a criação do study e a chamada optimize (15 trials)



# print(f"Melhor Acurácia: {study_guided.best_value:.4f}")
# print(f"Melhores Parâmetros: {study_guided.best_params}")


<details>
<summary>🔓 Clique para ver o gabarito (só depois de tentar!)</summary>

```python
def objective_guided(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    max_features = trial.suggest_float('max_features', 0.1, 1.0)

    model = ExtraTreesClassifier(
        n_estimators=n_estimators,
        criterion=criterion,
        max_features=max_features,
        random_state=42
    )
    model.fit(X_train_et, y_train_et)
    return accuracy_score(y_val_et, model.predict(X_val_et))

study_guided = optuna.create_study(direction='maximize')
study_guided.optimize(objective_guided, n_trials=15)
print(f"Melhor Acurácia: {study_guided.best_value:.4f}")
print(f"Melhores Parâmetros: {study_guided.best_params}")
```
</details>

### Nível 2: Onde Usar `log=True`? (Médio)

**Conceito didático:** quando um parâmetro pode variar em ordens de magnitude (ex: 0.0001 a 100), a distância linear não faz sentido — usamos `log=True` para que o Optuna explore cada ordem de grandeza igualmente.

Desta vez a decisão é sua: olhe os 3 hiperparâmetros do SVM abaixo e **decida sozinho** quais merecem `log=True` e quais não.

**Especificação da função `objective_wine(trial)`:**
1. Sugira `C` (float, entre `1e-5` e `1e2`) — pense: os valores possíveis vão de 0.00001 a 100. Faz sentido buscar linearmente aqui?
2. Sugira `gamma` (float, entre `1e-4` e `1e1`) — mesmo raciocínio do `C`.
3. Sugira `kernel` (categórico: `'linear'`, `'poly'`, `'rbf'`) — isso é uma escolha discreta, não numérica.
4. Crie um `SVC(C=..., gamma=..., kernel=..., random_state=42)` — note que `gamma` só é usado de fato pelos kernels `'poly'` e `'rbf'`, mas passá-lo sempre não quebra nada.
5. Treine e retorne a acurácia de validação.


In [ ]:
from sklearn.datasets import load_wine
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Dados para este exercício (já prontos)
wine = load_wine()
X_w, y_w = wine.data, wine.target
X_train_w, X_val_w, y_train_w, y_val_w = train_test_split(X_w, y_w, test_size=0.2, random_state=42)

# ESCREVA AQUI a função objective_wine(trial).
# Lembre-se de decidir, para CADA parâmetro numérico, se ele precisa de log=True.



# ESCREVA AQUI a criação do study e a otimização (20 trials)



# print(f"Melhor Acurácia: {study_wine.best_value:.4f}")
# print(f"Melhores Parâmetros: {study_wine.best_params}")


<details>
<summary>🔓 Clique para ver o gabarito</summary>

```python
def objective_wine(trial):
    c_value = trial.suggest_float('C', 1e-5, 1e2, log=True)       # log=True: varia em ordens de magnitude
    gamma_value = trial.suggest_float('gamma', 1e-4, 1e1, log=True)  # mesmo raciocínio
    kernel_choice = trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf'])  # categórico, sem log

    model = SVC(C=c_value, gamma=gamma_value, kernel=kernel_choice, random_state=42)
    model.fit(X_train_w, y_train_w)
    return accuracy_score(y_val_w, model.predict(X_val_w))

study_wine = optuna.create_study(direction='maximize')
study_wine.optimize(objective_wine, n_trials=20)
print(f"Melhor Acurácia: {study_wine.best_value:.4f}")
print(f"Melhores Parâmetros: {study_wine.best_params}")
```

**Por quê `log=True` em `C` e `gamma`, mas não faria sentido em `kernel`?** `kernel` é categórico — não existe "escala" entre `'linear'` e `'rbf'`, então `log` nem se aplica. Já `C` e `gamma` são contínuos e cobrem várias ordens de grandeza: sem `log=True`, o Optuna gastaria a maior parte das tentativas em valores grandes (ex: entre 50 e 100) e quase nunca testaria a faixa pequena (ex: entre 0.0001 e 0.001), que também pode conter o ótimo.
</details>

### Nível 3: Regressão com Validação Cruzada (Médio)

**Conceito didático:** nem todo problema é de classificação! Em regressão, normalmente **minimizamos** um erro (ex: MSE) em vez de maximizar uma acurácia. E, em vez de um único `train_test_split`, vamos usar **validação cruzada** (`cross_val_score`) dentro do `objective` — isso dá uma estimativa mais robusta do desempenho de cada trial.

**Especificação da função `objective_diabetes(trial)`:**
1. Sugira `n_estimators` (inteiro, entre 50 e 300).
2. Sugira `max_depth` (inteiro, entre 2 e 20).
3. Sugira `min_samples_leaf` (float, entre 0.01 e 0.3).
4. Crie um `RandomForestRegressor` com esses parâmetros e `random_state=42`.
5. Em vez de `.fit()` + `.predict()` manual, use `cross_val_score(model, X_diab, y_diab, cv=5, scoring='neg_mean_squared_error')` e tire a média com `.mean()`.
6. **Atenção:** `scoring='neg_mean_squared_error'` retorna o erro *negativo* (o sklearn sempre "maximiza" internamente). Você precisa decidir: retorna o valor como está, ou inverte o sinal? E o `direction` do `study` deve ser `'maximize'` ou `'minimize'`? Pense nisso antes de escrever.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Dados para este exercício (regressão) — já prontos
X_diab, y_diab = load_diabetes(return_X_y=True)

# ESCREVA AQUI a função objective_diabetes(trial).
# Dica: cross_val_score com scoring='neg_mean_squared_error' retorna valores
# NEGATIVOS de erro (quanto mais perto de 0, melhor). Decida como lidar com isso.



# ESCREVA AQUI a criação do study (escolha direction com cuidado!) e a otimização (25 trials)



# print(f"Melhor MSE: {study_diabetes.best_value:.4f}")
# print(f"Melhores Parâmetros: {study_diabetes.best_params}")


<details>
<summary>🔓 Clique para ver o gabarito</summary>

```python
def objective_diabetes(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 2, 20)
    min_samples_leaf = trial.suggest_float('min_samples_leaf', 0.01, 0.3)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    # neg_mean_squared_error vem negativo; multiplicamos por -1 para virar
    # o MSE "de verdade" (positivo), que é mais intuitivo de interpretar
    neg_mse = cross_val_score(model, X_diab, y_diab, cv=5, scoring='neg_mean_squared_error').mean()
    mse = -neg_mse
    return mse

# Como retornamos o MSE positivo (erro), queremos MINIMIZAR
study_diabetes = optuna.create_study(direction='minimize')
study_diabetes.optimize(objective_diabetes, n_trials=25)
print(f"Melhor MSE: {study_diabetes.best_value:.4f}")
print(f"Melhores Parâmetros: {study_diabetes.best_params}")
```

**A pegadinha:** se você retornasse `neg_mse` (sem inverter o sinal) e usasse `direction='minimize'`, o Optuna tentaria tornar o número negativo *ainda mais negativo* — ou seja, pioraria o modelo de propósito! Duas combinações funcionam: `mse` positivo + `minimize`, OU `neg_mse` como está + `maximize`. O que não pode é misturar os dois.
</details>

### Nível 4: Sampler Genético — TPE vs CMA-ES (Médio-Difícil)

Lembra da nota sobre **Algoritmos Genéticos** lá no Nível 0? Agora é a hora de usar de verdade. O Optuna troca de estratégia de busca através do parâmetro `sampler` no `create_study` — até aqui, sem você escolher, ele usava o **TPE** (Otimização Bayesiana) por padrão. Vamos comparar com o **CMA-ES**, um algoritmo evolutivo que mantém uma "população" de soluções que se adapta a cada geração.

Vamos resolver o **mesmo problema do Nível 1** (ExtraTrees no breast cancer), mas com dois samplers diferentes, para comparar.

**Especificação:**
1. Reaproveite a função `objective_guided` que você já escreveu no Nível 1 (mesmo código, sem mudar nada nela).
2. Crie `study_tpe` usando `sampler=optuna.samplers.TPESampler(seed=42)` explicitamente.
3. Crie `study_cmaes` usando `sampler=optuna.samplers.CmaEsSampler(seed=42)`.
4. Rode 30 trials em cada um e compare `best_value` e o tempo de execução.


In [ ]:
import time

# ESCREVA AQUI: crie study_tpe com sampler=optuna.samplers.TPESampler(seed=42),
# direction='maximize', rode 30 trials medindo o tempo com time.time()



# ESCREVA AQUI: crie study_cmaes com sampler=optuna.samplers.CmaEsSampler(seed=42),
# direction='maximize', rode 30 trials medindo o tempo



# print("=== TPE vs CMA-ES ===")
# print(f"TPE    -> score: {study_tpe.best_value:.4f}    | tempo: {t_tpe:.2f}s")
# print(f"CMA-ES -> score: {study_cmaes.best_value:.4f} | tempo: {t_cmaes:.2f}s")


<details>
<summary>🔓 Clique para ver o gabarito</summary>

```python
import time

study_tpe = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
t0 = time.time()
study_tpe.optimize(objective_guided, n_trials=30)
t_tpe = time.time() - t0

study_cmaes = optuna.create_study(direction='maximize', sampler=optuna.samplers.CmaEsSampler(seed=42))
t0 = time.time()
study_cmaes.optimize(objective_guided, n_trials=30)
t_cmaes = time.time() - t0

print("=== TPE vs CMA-ES ===")
print(f"TPE    -> score: {study_tpe.best_value:.4f}    | tempo: {t_tpe:.2f}s")
print(f"CMA-ES -> score: {study_cmaes.best_value:.4f} | tempo: {t_cmaes:.2f}s")
```

**O que observar:** para espaços de busca pequenos e baratos como este, os dois costumam empatar em qualidade. A diferença aparece em espaços **grandes e complexos**, onde o CMA-ES tende a se sair melhor por manter uma "população" que explora várias regiões ao mesmo tempo — mas ao custo de precisar avaliar essa população inteira a cada geração, o que é mais caro computacionalmente. Na dúvida, o TPE (padrão do Optuna) é a escolha mais segura para a maioria dos problemas do dia a dia.
</details>

### Nível 5: Espaço de Busca Condicional — Define-by-Run de Verdade (Difícil)

Até aqui, todo `objective` sugeria os mesmos parâmetros sempre. Mas o Optuna é **define-by-run**: o espaço de busca pode mudar *dentro* da própria execução do trial, com `if`s normais de Python. É isso que o diferencia do Grid Search, que precisa de um espaço de busca fixo definido antes.

**Especificação da função `objective_gb(trial)`:**
1. Sugira `learning_rate` (float, entre `1e-3` e `0.3`, com `log=True`).
2. Sugira `n_estimators` (inteiro, entre 50 e 300).
3. Sugira `max_depth` (inteiro, entre 2 e 10).
4. Sugira uma escolha categórica `limit_leaves` entre `[True, False]`.
5. **Aqui está o pulo do gato:** SE `limit_leaves` for `True`, sugira também `max_leaf_nodes` (inteiro, entre 10 e 50) e passe esse parâmetro pro modelo. SE for `False`, não sugira `max_leaf_nodes` — nem crie a variável — e não passe esse parâmetro pro modelo (deixe o valor padrão do sklearn).
6. Crie o `GradientBoostingClassifier` com os parâmetros — atenção, o dicionário de parâmetros vai ter tamanhos diferentes dependendo do caminho do `if`.
7. Treine com `X_train_et, y_train_et` e retorne a acurácia em `X_val_et, y_val_et`.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# ESCREVA AQUI a função objective_gb(trial) seguindo a especificação acima.
# Dica: monte um dicionário de kwargs e use **kwargs ao criar o modelo,
# ou simplesmente crie o modelo dentro de cada branch do if/else.



# study_gb = optuna.create_study(direction='maximize')
# study_gb.optimize(objective_gb, n_trials=30)
# print(f"Melhor Acurácia: {study_gb.best_value:.4f}")
# print(f"Melhores Parâmetros: {study_gb.best_params}")


<details>
<summary>🔓 Clique para ver o gabarito</summary>

```python
def objective_gb(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 2, 10)

    limit_leaves = trial.suggest_categorical('limit_leaves', [True, False])

    params = dict(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Espaço de busca CONDICIONAL: max_leaf_nodes só existe se limit_leaves=True.
    # Isso é o "define-by-run": a lógica normal do Python decide o espaço de busca.
    if limit_leaves:
        params['max_leaf_nodes'] = trial.suggest_int('max_leaf_nodes', 10, 50)

    model = GradientBoostingClassifier(**params)
    model.fit(X_train_et, y_train_et)
    return accuracy_score(y_val_et, model.predict(X_val_et))

study_gb = optuna.create_study(direction='maximize')
study_gb.optimize(objective_gb, n_trials=30)
print(f"Melhor Acurácia: {study_gb.best_value:.4f}")
print(f"Melhores Parâmetros: {study_gb.best_params}")
```

**Por que isso importa:** um `GridSearchCV` clássico não consegue fazer isso de forma limpa — você teria que rodar duas grades separadas e juntar os resultados. Com o Optuna, a lógica condicional é só Python normal dentro da função. É a essência do "define-by-run" citado nos slides.
</details>

### Nível 6: XGBoost + Pruning, do Zero (⚡ O Mais Desafiador)

Este é o exercício final e o mais completo. Ele reúne **tudo** que você viu até aqui — inteiro, float, log-scale — e adiciona um conceito novo e avançado: **pruning** (poda). Desta vez não tem gabarito colado ao lado: você escreve o loop de treino incremental e a lógica de poda do zero, só com a especificação.

**Conceito — Pruning:** em modelos que treinam em rounds (como o boosting), não faz sentido esperar até o fim se o trial já está claramente pior que os anteriores. A cada bloco de rounds, reportamos o desempenho parcial com `trial.report(valor, step)` e perguntamos `trial.should_prune()`. Se for `True`, abortamos o trial levantando `optuna.TrialPruned()` — e o Optuna já sabe descartar esse resultado e seguir para o próximo trial.

**Especificação da função `final_objective(trial)`:**
1. Sugira `learning_rate` (float, entre `1e-3` e `0.1`, `log=True` — é a taxa de aprendizado do boosting).
2. Sugira `max_depth` (inteiro, entre 3 e 9).
3. Sugira `subsample` (float, entre 0.5 e 1.0 — fração de linhas usada em cada árvore).
4. Sugira `colsample_bytree` (float, entre 0.5 e 1.0 — fração de features usada em cada árvore).
5. Defina `n_estimators = 200` (fixo, não é hiperparâmetro sugerido — é o total de rounds de boosting que vamos treinar incrementalmente).
6. Crie um `xgb.XGBClassifier` com esses hiperparâmetros, `random_state=42` e `eval_metric='logloss'`.
7. **O loop de treino incremental e pruning** — este é o cerne do exercício:
   - Defina `step_size = 20`.
   - Faça um loop de `i` indo de `step_size` até `n_estimators` (inclusive), pulando de `step_size` em `step_size` (dica: `range(step_size, n_estimators + 1, step_size)`).
   - A cada iteração, ajuste o modelo para treinar até `i` árvores (`model.set_params(n_estimators=i)`) e treine incrementalmente passando `xgb_model=model.get_booster()` a partir da segunda iteração em diante (na primeira, `xgb_model=None`).
   - Calcule a acurácia de validação nesse ponto intermediário.
   - Reporte esse valor pro Optuna com `trial.report(...)`, passando `i` como `step`.
   - Cheque `trial.should_prune()`: se `True`, levante `optuna.TrialPruned()` para abortar o trial imediatamente.
8. Depois do loop, retorne a última acurácia calculada.
9. Crie o `study` com `direction='maximize'` e `pruner=optuna.pruners.MedianPruner()`, e rode 30 trials.
10. Ao final, imprima também quantos trials foram efetivamente podados (dica: `trial.state.name == 'PRUNED'` para cada `trial` em `study_final.trials`).


In [ ]:
import xgboost as xgb

# ESCREVA AQUI a função final_objective(trial) seguindo a especificação acima,
# passo a passo. É normal essa ser a célula mais longa do notebook — vá com calma.



# ESCREVA AQUI a criação do study com pruner=optuna.pruners.MedianPruner()
# e a chamada optimize (30 trials)



# print(f"Melhor Acurácia: {study_final.best_value:.4f}")
# print(f"Melhores Parâmetros: {study_final.best_params}")
# print(f"Trials podados: {len([t for t in study_final.trials if t.state.name == 'PRUNED'])} de {len(study_final.trials)}")


<details>
<summary>🔓 Clique para ver o gabarito (tente escrever antes — este é o exercício mais importante do notebook!)</summary>

```python
def final_objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 9)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)

    n_estimators = 200  # total de rounds de boosting

    model = xgb.XGBClassifier(
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        n_estimators=n_estimators,
        random_state=42,
        eval_metric='logloss'
    )

    # PRUNING: treinamos incrementalmente e checamos a cada bloco de rounds
    # se vale a pena continuar.
    step_size = 20
    for i in range(step_size, n_estimators + 1, step_size):
        model.set_params(n_estimators=i)
        model.fit(X_train_et, y_train_et, xgb_model=None if i == step_size else model.get_booster())
        intermediate_acc = accuracy_score(y_val_et, model.predict(X_val_et))

        trial.report(intermediate_acc, step=i)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return intermediate_acc

study_final = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
study_final.optimize(final_objective, n_trials=30)

print(f"Melhor Acurácia: {study_final.best_value:.4f}")
print(f"Melhores Parâmetros: {study_final.best_params}")
print(f"Trials podados: {len([t for t in study_final.trials if t.state.name == 'PRUNED'])} de {len(study_final.trials)}")
```

**Por que treinar incrementalmente com `xgb_model=model.get_booster()`?** Sem isso, cada iteração do loop treinaria um modelo do zero com `i` árvores — desperdiçando todo o trabalho feito até ali. Passando o booster anterior, o XGBoost **continua** de onde parou, adicionando só as `step_size` árvores novas. É isso que torna o pruning eficiente: paramos de treinar cedo sem ter perdido o progresso já feito.
</details>

---

Chegou até aqui e escreveu tudo do zero? Você aprendeu o essencial do Optuna: `objective`, os `suggest_*`, `study` com `direction`, samplers alternativos e pruning. Isso é o suficiente pra aplicar em praticamente qualquer projeto de ML.


## 5. Visualização e Diagnóstico

Após a otimização, não queremos apenas o "melhor número". Queremos entender o modelo. O Optuna oferece:
* **Optimization History**: mostra se a busca está convergindo.
* **Param Importance**: revela quais variáveis realmente mudam o jogo.
* **Slice Plots**: mostram a relação entre um parâmetro específico e o resultado.


In [ ]:
# Usando o study do Nível 6 (XGBoost + Pruning) como exemplo, por ser o mais rico

# Histórico de Otimização
optuna.visualization.plot_optimization_history(study_final).show()

# Importância dos Hiperparâmetros
# (pode falhar se houver pouca variância entre os trials completos — ex: muitos
# trials podados deixam poucos pontos "cheios" para calcular importância)
try:
    optuna.visualization.plot_param_importances(study_final).show()
except RuntimeError as e:
    print(f"Não foi possível calcular a importância dos parâmetros: {e}")
    print("Isso pode acontecer quando há poucos trials completos (não podados) ou pouca variação entre eles.")

# Gráfico de Fatias (Slice Plot)
optuna.visualization.plot_slice(study_final).show()


**O VEREDITO:** olhe o gráfico de importância acima e veja qual parâmetro teve a barra mais longa. Isso diz a você: "para este problema, o que mais importou foi X".


## 6. Para Que Servem os Hiperparâmetros na Prática?

Otimizar hiperparâmetros não é só sobre "ganhar acurácia". Eles têm papéis diferentes dependendo do que você precisa resolver:

**1. Velocidade de treino/inferência**
Às vezes você ajusta hiperparâmetros para ganhar tempo, mesmo abrindo mão de um pouco de performance — por exemplo, reduzir `n_estimators` ou `max_depth` para um modelo que precisa responder em produção com baixa latência.

**2. Dados desbalanceados**
Hiperparâmetros não têm necessariamente a ver com complexidade do modelo — alguns fazem o modelo prestar mais atenção na classe minoritária, quando o dataset tem muito mais exemplos de uma classe do que de outra (ex: `class_weight`, `scale_pos_weight`).

**3. Velocidade de convergência**
Afeta quantas iterações (`n_estimators`) você vai precisar até o modelo estabilizar. Um `learning_rate` maior converge mais rápido, mas com mais risco de "atropelar" o ótimo.

**4. Uso de memória/recursos computacionais**
Hiperparâmetros como `max_depth` ou `n_estimators` também controlam quanto de RAM/processamento o modelo consome — relevante quando você está treinando em ambientes com recursos limitados.

Tenha esses 4 pontos em mente: otimizar hiperparâmetros é sempre um trade-off entre **performance**, **tempo** e **recursos** — não existe "o melhor hiperparâmetro" no vácuo, existe o melhor para o seu problema específico.


## 7. Conclusão e Resumo Final

A otimização não é apenas "rodar o código". Um bom cientista de dados observa os gráficos de importância para entender se o espaço de busca estava correto.

* **Dica de Ouro**: se o melhor valor estiver no limite da sua busca (ex: você buscou até 100 e o melhor foi 100), aumente o limite e rode novamente!
* **Eficiência**: use o Optuna para economizar tempo de computação e focar na análise dos dados.

### Cola rápida dos métodos `suggest`

| Método | Uso | Exemplo |
|---|---|---|
| `suggest_int` | Números inteiros | `trial.suggest_int('n_estimators', 10, 200)` |
| `suggest_float` | Números decimais | `trial.suggest_float('learning_rate', 1e-3, 0.1, log=True)` |
| `suggest_categorical` | Escolhas discretas (strings/opções) | `trial.suggest_categorical('kernel', ['linear', 'rbf'])` |
| `log=True` | Para parâmetros que variam em ordens de magnitude (ex: 0.001 a 100) | — |

### Fim do Guia Prático 🎉
